In [3]:
import pandas as pd
import re

In [ ]:
https://www.bda.gv.at/service/unterschutzstellung/denkmalverzeichnis/denkmalliste-gemaess-3-dmsg.html
D7 = ("datasets/upperaustria_denkmal_list.csv")

In [4]:
denkmal = pd.read_csv(D7, sep=";")
denkmal.head()

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/upperaustria_denkmal_list.csv'

In [5]:
# Keyword filter: keep monuments a visitor would actually go and see.
# Categories are checked in order, the first match wins.
CATEGORIES = {
    "museum_memorial": (
        r"(?<!ehem\. )museum|gedenkstätte|memorial|besucherzentrum"
        r"|konzentrationslager|kz-|geburtshaus|sterbehaus"
    ),
    "abbey": (
        r"\bstift\b|stiftskirche|stiftspfarrkirche|kloster|abtei"
        r"|basilika|\bdom\b|domkirche"
    ),
    "castle": (
        r"schloss|schloß|schlössl|\bburg\b|burgruine|herzogsburg"
        r"|\bruine|festung|palais|residenz|\bansitz"
    ),
    "archaeology": (
        r"römer|römisch|lauriacum|legionslager|kastell|vicus|fundzone"
        r"|gräberfeld|hügelgr|ringwall|pfahlbau|höhensiedlung"
        r"|burgstall|hallstattkultur|felszeichnung|befunde"
    ),
    "pilgrimage": (r"wallfahrt"),
    "fortification": (
        r"stadtmauer|marktmauer|befestigung|stadttor|\btor\b|torturm"
        r"|torbau|bastion|zwinger|wehrturm|\bturm\b|stadtturm|\bfort\b"
    ),
    "landmark": (
        r"rathaus|pranger|mariensäule|pestsäule|dreifaltigkeitssäule"
        r"|marktbrunnen|stadtbrunnen|(?<!krieger)denkmal|lichtsäule"
        r"|totenleuchte"
    ),
    "technical": (
        r"soleleitung|hammer|sensenwerk|pferdeeisenbahn|seilbahn"
        r"|lokalbahn|pöstlingbergbahn|klause|glockengießerei"
        r"|salzstadel|salzturm|innerberger stadel|stollen"
    ),
    "culture_venue": (
        r"theater|brucknerhaus|kurhaus|trinkhalle|kaiservilla"
        r"|musikpavillon|botanischer garten|lehár|stadthalle"
    ),
}

# If the main (first) part of the title is an everyday building, drop it,
# even when a later part says e.g. "ehem. Schloss" or "Teil der Stadtmauer".
EXCLUDE_LEAD = (
    r"^(ehem\.\s+|alte[rs]?\s+)?(bürger|wohn|miet|doppelwohn"
    r"|arbeiter|kommunal|ackerbürger|pfarrhof|pfarrheim|pfarrhaus"
    r"|pfarrzentrum|stadtpfarrhof|dechanthof|mesner|benefiziat"
    r"|kaplan|.*schule\b|.*gymnasium|kindergarten"
    r"|(markt)?gemeindeamt|amtsgebäude|bank|sparkasse|nationalbank"
    r"|bezirksgericht|kreisgericht|bezirkshauptmannschaft|bh\s"
    r"|amt der|landesregierung|.*kammer|kaserne|krankenhaus"
    r"|wirtschaftsgebäude|bauern|meierhof|stiftsmeierhof"
    r"|.*stiftshaus|stiftshof|freihaus|erdkeller|keller|gasthaus"
    r"|gasthof|hotel|lagergebäude|verwaltung|forsthaus|forsthube"
    r"|stadel|schüttkasten|friedhof|aufbahrungshalle|grenzstein"
    r"|.*bildstock|figur|kruzifix|kriegerdenkmal|gedenk(tafel"
    r"|stein)|wegkapelle|flur|hofkapelle|nischenkapelle|kath\."
    r"|evang\.|\w*kirche|\w*kapelle|karner|kalvarienberg"
    r"|\w*kreuzweg)"
)

# Always keep these, whatever the first part of the title says
# (e.g. "Wohnhaus, Bruckner-Geburtshaus", "Troadkasten, Heimatmuseum",
# "Kath. Pfarrkirche, Wallfahrtskirche ...").
ALWAYS_KEEP = (
    r"(?<!ehem\. )museum|gedenkstätte|memorial|konzentrationslager"
    r"|kz-|geburtshaus|wallfahrt|basilika|\bdom\b|stiftskirche"
    r"|stiftspfarrkirche|klosterkirche|klosterbauten"
)


def classify(title: object) -> str | None:
    t = str(title).lower()
    lead = t.split(",")[0].strip()
    if not re.search(ALWAYS_KEEP, t) and re.search(EXCLUDE_LEAD, lead):
        return None
    for category, pattern in CATEGORIES.items():
        if re.search(pattern, t):
            return category
    return None


denkmal["category"] = denkmal["Objekttitel"].map(classify)
touristic = denkmal[denkmal["category"].notna()].copy()

NameError: name 'denkmal' is not defined

In [ ]:
# Clean addresses for geocoding: drop the "#MIG-x#" markers and the
"""
position hints like "(bei)", "(vor)", "(gegenüber)".
POSITION_HINT = r"\s*\((bei|vor|gegenüber|hinter|neben|in der nähe)\)"

touristic["address_clean"] = (
    touristic["Adresse"]
    .str.replace(r"\s*#MIG-\w+#", "", regex=True)
    .str.replace(POSITION_HINT, "", regex=True, flags=re.IGNORECASE)
    .str.replace(r"\s+,", ",", regex=True)
    .str.strip()
 )

touristic = touristic.dropna(subset=["Adresse"])
print(len(touristic), "of", len(denkmal), "monuments kept")
print(touristic["category"].value_counts())

touristic[["HERIS-ID", "Gemeinde", "Objekttitel", "category", "Adresse"]].to_csv(
    "datasets/upperaustria_denkmal_touristic.csv",
    sep=";",
    index=False,
)
"""

,HERIS-ID,Bundesland,Gemeinde,KG,Objekttitel,Adresse,Grundstücksnummern,Denkmalschutz Status,category
1,18696,Oberösterreich,Adlwang,49001 Adlwang,"Kath. Filialkirche, Wallfahrtskirche St. Blasien","Grünburger Straße 117 , 4541 Adlwang",.66,Denkmalschutz per Verordnung,pilgrimage
3,18677,Oberösterreich,Adlwang,49001 Adlwang,"Kath. Pfarrkirche, Wallfahrtskirche Sieben Sch...","Kirchenplatz 1 , 4541 Adlwang",.1,Denkmalschutz per Verordnung,pilgrimage
13,18456,Oberösterreich,Aigen-Schlägl,47001 Aigen,Mariensäule,"Marktplatz 21 , 4160 Aigen-Schlägl (Aigen im M...",2860/1,Denkmalschutz per Verordnung,landmark
19,20149,Oberösterreich,Aigen-Schlägl,47009 Schlägl,"Sommerhaus Stift Schlägl, Musikzentrum St. Nor...","Schaubergstraße 7 , 4160 Aigen-Schlägl (Schlägl)",59,Denkmalschutz per Verordnung,abbey
20,20144,Oberösterreich,Aigen-Schlägl,47009 Schlägl,Prämonstratenserkloster Stift Schlägl,"Schlägl 1 , 4160 Aigen-Schlägl",".20, .21",Denkmalschutz per Verordnung,abbey
...,...,...,...,...,...,...,...,...,...
5933,38912,Oberösterreich,Windischgarsten,49412 Windischgarsten,Römischer Gebäudekomplex,"Kirchfeldstraße 26 , 4580 Windischgarsten (bei)","945, 947/3",Denkmalschutz per Bescheid,archaeology
5936,81168,Oberösterreich,Windischgarsten,49412 Windischgarsten,Heimatmuseum,"Museumstraße 7 , 4580 Windischgarsten",.1,Denkmalschutz per Verordnung,museum_memorial
5945,38268,Oberösterreich,Wolfern,49216 Losensteinleithen,Schloss Losensteinleithen,"Losensteinleiten 1 , 4493 Wolfern (Losensteinl...",10/5,Denkmalschutz per Bescheid (Unterschutzstellun...,castle
5956,38923,Oberösterreich,Wolfsegg am Hausruck,50217 Wolfsegg,Schloss Wolfsegg mit Wirtschaftsgebäude,"Schloßberg 1 , 4902 Wolfsegg am Hausruck",".1, .2/1",Denkmalschutz per Bescheid (Unterschutzstellun...,castle


In [6]:
touristic.to_csv("filtered_touristic.csv", sep=";")